In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, SimpleRNN
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

In [13]:
X = np.loadtxt('../data/processed/X_7steps_encoded.csv', delimiter=',')
y = np.loadtxt('../data/processed/y_7steps_encoded.csv', delimiter=',')

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (211, 131)
y shape: (211,)


In [15]:
X_new = np.delete(X, [1, 2], axis=1)
X_new.shape

(211, 129)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.1, random_state=5)

print('X Train shape:', X_train.shape)
print('X Test shape:', X_test.shape)
print('y Train shape:', y_train.shape)
print('y Test shape:', y_test.shape)

X Train shape: (189, 129)
X Test shape: (22, 129)
y Train shape: (189,)
y Test shape: (22,)


In [17]:
ann = Sequential([
  Input(shape=(X_train.shape[1],)),
  Dense(32, activation='relu'),
  Dense(128, activation='softmax')
])
ann.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 32)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │         4,224 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,384 (32.75 KB)

 Trainable params: 8,384 (32.75 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
ann.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(), metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)])
ann.fit(X_train, y_train, epochs=100, validation_split=0.2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.0254 - loss: 4.8381 - sparse_top_k_categorical_accuracy: 0.0720 - val_accuracy: 0.0000e+00 - val_loss: 4.8441 - val_sparse_top_k_categorical_accuracy: 0.0263
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0366 - loss: 4.7848 - sparse_top_k_categorical_accuracy: 0.1280 - val_accuracy: 0.0000e+00 - val_loss: 4.8001 - val_sparse_top_k_categorical_accuracy: 0.0789
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0372 - loss: 4.7405 - sparse_top_k_categorical_accuracy: 0.2241 - val_accuracy: 0.0000e+00 - val_loss: 4.7560 - val_sparse_top_k_categorical_accuracy: 0.1579
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0916 - loss: 4.6871 - sparse_top_k_categorical_accuracy: 0.3497 - val_accuracy: 0.0000e+00 - val_loss: 4.7099 - val_sparse_top_k_categorical_accuracy: 0.2895
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1624 - loss: 4.6188 - sparse_top_k_categorical_a

In [19]:
ann.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 529ms/step - accuracy: 0.2273 - loss: 2.3099 - sparse_top_k_categorical_accuracy: 0.6818


[2.309852361679077, 0.22727273404598236, 0.6818181872367859]

### Tuning Hyperparameters

In [20]:
def build_model(hp):
  model = Sequential()
  model.add(Input(shape=(129,)))

  # Hidden Layer 1
  model.add(Dense(
    units=hp.Int('units1', min_value=16, max_value=128, step=16),
    activation='relu',
  ))

  # Dropout Layer 1 (Optional)
  if hp.Boolean('use_dropout1_layer'):
    model.add(Dropout(hp.Float('dropout1', min_value=0.1, max_value=0.5, step=0.1)))

  # Hidden Layer 2 (Optional)
  if hp.Boolean('use_hidden_layer2'):
    model.add(Dense(
      units=hp.Int('units2', min_value=16, max_value=128, step=16),
      activation='relu'
    ))

  # Dropout Layer 2 (Optional)
  if hp.Boolean('use_dropout2_layer'):
    model.add(Dropout(hp.Float('dropout2', min_value=0.1, max_value=0.5, step=0.1)))

  # Output Layer
  model.add(Dense(128, activation='softmax'))

  # Optimizer and Learning Rate
  lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
  optimizer = Adam(learning_rate=lr)

  model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)]
  )

  return model

In [21]:
tuner = kt.RandomSearch(
  build_model,
  objective='val_accuracy',
  max_trials=20,
  executions_per_trial=1,
  directory='../tuning_dir',
  project_name='hero_patch_version'
)

tuner.search(X_train, y_train, epochs=100, validation_split=0.2)

Trial 20 Complete [00h 00m 25s]
val_accuracy: 0.34210526943206787

Best val_accuracy So Far: 0.5263158082962036
Total elapsed time: 00h 10m 22s


In [22]:
best_model = tuner.get_best_models()[0]
best_hps = tuner.get_best_hyperparameters()[0]

print("Best Hyperparameters:")
print(f"Units1: {best_hps['units1']}")
print(f"Use Dropout1 Layer: {best_hps['use_dropout1_layer']}")
if best_hps['use_dropout1_layer']:
    print(f"Dropout1: {best_hps['dropout1']}")
print(f"Use Second Layer: {best_hps['use_hidden_layer2']}")
if best_hps['use_hidden_layer2']:
    print(f"Units2: {best_hps['units2']}")
print(f"Use Dropout2 Layer: {best_hps['use_dropout2_layer']}")
if best_hps['use_dropout2_layer']:
    print(f"Dropout2: {best_hps['dropout2']}")
print(f"Learning Rate: {best_hps['learning_rate']}")

Best Hyperparameters:
Units1: 112
Use Dropout1 Layer: True
Dropout1: 0.1
Use Second Layer: True
Units2: 80
Use Dropout2 Layer: False
Learning Rate: 0.01


c:\Users\segav\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\saving\saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [23]:
best_model.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3636 - loss: 4.2502 - sparse_top_k_categorical_accuracy: 0.7273


[4.250194072723389, 0.3636363744735718, 0.7272727489471436]

In [24]:
best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]

print("Best trial ID:", best_trial.trial_id)
print("Best val_accuracy:", best_trial.score)
print("Best hyperparameters:", best_trial.hyperparameters.values)


Best trial ID: 01
Best val_accuracy: 0.5263158082962036
Best hyperparameters: {'units1': 112, 'use_dropout1_layer': True, 'use_hidden_layer2': True, 'use_dropout2_layer': False, 'learning_rate': 0.01, 'dropout1': 0.1, 'units2': 80, 'dropout2': 0.1}
